# Pipeline Automation – Run Notebook
Esegui le celle in sequenza per completare l'intera pipeline end-to-end:
RAG Generation → Blind Tournament → DPO Preferences Builder → Analisi & Grafici.

## Cella 1: Configurazione Globale dei Parametri e Check Hardware GPU
*In questa cella imposti le variabili di controllo e verifichi che PyTorch veda correttamente la tua RTX 4070 su Windows.*

In [ ]:
import os
import torch

# === CONFIGURAZIONE PARAMETRI ===
INPUT_DATA = "data/raw/task-a-en.tsv"
RAG_CONFIG = "configs/rag.yaml"

# PARAMETRO DI TEST: Lascia "--limit 20" per testare la pipeline in pochi minuti.
# Sostituisci con "" (stringa vuota) quando vuoi lanciare la produzione sull'intero dataset.
LIMIT = "--limit 20"

# Inserisci qui il tuo token Hugging Face per scaricare i modelli protetti
os.environ["HF_TOKEN"] = "token"

# === CHECK HARDWARE GPU ===
gpu_disponibile = torch.cuda.is_available()
print(f"Configurazione iniziale completata.\n")
print(f"GPU Disponibile per PyTorch: {gpu_disponibile}")

if gpu_disponibile:
    print(f"Nome della scheda video rilevata: {torch.cuda.get_device_name(0)}")
    print(f"Versione CUDA di PyTorch: {torch.version.cuda}")
    print(f"\n\U0001f680 Schedulazione impostata su: {'MODALITÀ TEST (20 righe)' if LIMIT else 'PRODUCTION (Dataset intero)'}")
else:
    print("\u26a0\ufe0f Attenzione: PyTorch non vede la GPU! Controlla i driver NVIDIA CUDA su Windows.")

Configurazione iniziale completata.

GPU Disponibile per PyTorch: True
Nome della scheda video rilevata: NVIDIA GeForce RTX 4070
Versione CUDA di PyTorch: 12.4

🚀 Schedulazione impostata su: MODALITÀ TEST (20 righe)


## Cella 2: Fase 1 – RAG Generation con LLAMA
*Avvia la generazione di joke reali con contesto Wikipedia sfruttando Llama 3.1 8B.*

In [6]:
%%time
MODEL_1 = "llama"
OUTPUT_LLAMA = f"data/generated/rag/{MODEL_1}_rag.jsonl"

print(f"--- Avvio Generazione RAG con {MODEL_1.upper()} ---")
!python scripts/run_rag_generate.py \
  --model {MODEL_1} \
  --input {INPUT_DATA} \
  --output {OUTPUT_LLAMA} \
  --rag-config {RAG_CONFIG} \
  --k 4 \
  --apply-to headline \
  --overwrite \
  {LIMIT}

print(f"\n\u2713 Output Llama salvato con successo in: {OUTPUT_LLAMA}")

--- Avvio Generazione RAG con LLAMA ---

✓ Output Llama salvato con successo in: data/generated/rag/llama_rag.jsonl
CPU times: total: 2.06 s
Wall time: 15min 1s


2026-06-25 12:02:05,390 | INFO | Loading SentenceTransformer model from mixedbread-ai/mxbai-embed-large-v1.
2026-06-25 12:02:08,224 | INFO | Loaded 1 prompt with these keys: ['query']
2026-06-25 12:02:08,254 | INFO | Loading unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\peppe\LLM-Project-SemEval-Humor-Generation\.venv\Lib\site-packages\transformers\quantizers\auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
2026-06-25 12:02:09,954 | INFO | We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).

Generating llama/rag:   0%|          | 0/20 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00

## Cella 3: Fase 1 – RAG Generation con QWEN
*Avvia lo stesso processo sequenziale per il secondo modello della matrice (Qwen 2.5 7B).*

In [7]:
%%time
MODEL_2 = "qwen"
OUTPUT_QWEN = f"data/generated/rag/{MODEL_2}_rag.jsonl"

print(f"--- Avvio Generazione RAG con {MODEL_2.upper()} ---")
!python scripts/run_rag_generate.py \
  --model {MODEL_2} \
  --input {INPUT_DATA} \
  --output {OUTPUT_QWEN} \
  --rag-config {RAG_CONFIG} \
  --k 4 \
  --apply-to headline \
  --overwrite \
  {LIMIT}

print(f"\n\u2713 Output Qwen salvato con successo in: {OUTPUT_QWEN}")

--- Avvio Generazione RAG con QWEN ---

✓ Output Qwen salvato con successo in: data/generated/rag/qwen_rag.jsonl
CPU times: total: 531 ms
Wall time: 7min 10s


2026-06-25 12:39:19,711 | INFO | Loading SentenceTransformer model from mixedbread-ai/mxbai-embed-large-v1.
2026-06-25 12:39:22,554 | INFO | Loaded 1 prompt with these keys: ['query']
2026-06-25 12:39:22,577 | INFO | Loading unsloth/Qwen2.5-7B-Instruct-bnb-4bit
`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\peppe\LLM-Project-SemEval-Humor-Generation\.venv\Lib\site-packages\transformers\quantizers\auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
2026-06-25 12:39:23,894 | INFO | We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).

Generating qwen/rag:   0%|          | 0/20 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:0

## Cella 4: Fase 1 – RAG Generation con MISTRAL
*Completa la tripletta di modelli della griglia avviando Mistral 7B.*

In [8]:
%%time
MODEL_3 = "mistral"
OUTPUT_MISTRAL = f"data/generated/rag/{MODEL_3}_rag.jsonl"

print(f"--- Avvio Generazione RAG con {MODEL_3.upper()} ---")
!python scripts/run_rag_generate.py \
  --model {MODEL_3} \
  --input {INPUT_DATA} \
  --output {OUTPUT_MISTRAL} \
  --rag-config {RAG_CONFIG} \
  --k 4 \
  --apply-to headline \
  --overwrite \
  {LIMIT}

print(f"\n\u2713 Output Mistral salvato con successo in: {OUTPUT_MISTRAL}")

--- Avvio Generazione RAG con MISTRAL ---

✓ Output Mistral salvato con successo in: data/generated/rag/mistral_rag.jsonl
CPU times: total: 2.11 s
Wall time: 17min 42s


2026-06-25 12:46:43,510 | INFO | Loading SentenceTransformer model from mixedbread-ai/mxbai-embed-large-v1.
2026-06-25 12:46:46,447 | INFO | Loaded 1 prompt with these keys: ['query']
2026-06-25 12:46:46,472 | INFO | Loading unsloth/mistral-7b-instruct-v0.3-bnb-4bit
`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\peppe\LLM-Project-SemEval-Humor-Generation\.venv\Lib\site-packages\transformers\quantizers\auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
2026-06-25 12:46:47,886 | INFO | We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).

Generating mistral/rag:   0%|          | 0/20 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00

## Cella 5: Fase 2 – Blind Tournament (Giudice Qwen)
*Mette a confronto riga per riga i file prodotti dai vari modelli per raccogliere i verdetti del torneo alla cieca.*

In [13]:
%%time
INPUT_DIR_JUDGE = "data/generated/rag"
OUTPUT_JUDGMENTS = "data/judged/rag_test_judgments.jsonl"

print("--- Avvio Torneo Blind tra i Modelli Generati ---")
# NOTA DI SICUREZZA: Assicurati di aver aggiunto il .replace('"', "'") per joke_a e joke_b
# nel file src/humor_gen/judge.py prima di eseguire questa cella,
# così eviterai crash sul parser JSON!
!python scripts/run_judge.py \
  --input-dir {INPUT_DIR_JUDGE} \
  --output {OUTPUT_JUDGMENTS} \
  --method rag \
  --overwrite

print(f"\n\u2713 Torneo concluso. Verdetto salvato in: {OUTPUT_JUDGMENTS}")

--- Avvio Torneo Blind tra i Modelli Generati ---

⚠️ [Tentativo 1/3] Errore JSON dal Giudice. Rigenero la risposta...

✓ Torneo concluso. Verdetto salvato in: data/judged/rag_test_judgments.jsonl
CPU times: total: 1.59 s
Wall time: 14min 50s


2026-06-25 14:59:08,015 | INFO | Loading unsloth/mistral-7b-instruct-v0.3-bnb-4bit
`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\peppe\LLM-Project-SemEval-Humor-Generation\.venv\Lib\site-packages\transformers\quantizers\auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
2026-06-25 14:59:09,780 | INFO | We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).

Judging llama vs qwen by mistral: 100%|██████████| 20/20 [04:43<00:00, 14.19s/it]
2026-06-25 15:03:56,136 | INFO | Loading unsloth/Qwen2.5-7B-Instruct-bnb-4bit
2026-06-25 15:03:57,577 | INFO | We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer 

## Cella 6: Fase 3 – Estrazione delle Preferenze (DPO Dataset Builder)
*Analizza i giudizi emessi dal torneo nella cella precedente e isola le risposte vincenti (chosen) e scartate (rejected).*

In [14]:
%%time
INPUT_JUDGMENTS = "data/judged/rag_test_judgments.jsonl"
OUTPUTS_DIR_PREF = "data/generated/rag"
OUTPUT_PREFERENCES = "data/final/test_preferences.jsonl"

print("--- Estrazione coppie di preferenze Chosen/Rejected (DPO Builder) ---")
!python scripts/build_preferences.py \
  --judgments {INPUT_JUDGMENTS} \
  --outputs-dir {OUTPUTS_DIR_PREF} \
  --output {OUTPUT_PREFERENCES} \
  --overwrite

print(f"\n\u2713 Dataset DPO strutturato e pronto in: {OUTPUT_PREFERENCES}")

--- Estrazione coppie di preferenze Chosen/Rejected (DPO Builder) ---

✓ Dataset DPO strutturato e pronto in: data/final/test_preferences.jsonl
CPU times: total: 15.6 ms
Wall time: 184 ms


## Cella 7: Fase 4 – Analisi Metriche e Generazione Report Grafici
*Aggrega le statistiche finali di vincita ed esporta i grafici delle performance della pipeline.*

In [15]:
%%time
GENERATED_DIR = "data/generated/rag"
JUDGED_DIR = "data/judged"
REPORT_FIGURES_DIR = "reports/figures"

print("--- Generazione Report Analitici e Grafici delle Performance ---")
!python scripts/analyze_results.py \
  --generated-dir {GENERATED_DIR} \
  --judged-dir {JUDGED_DIR} \
  --output-dir {REPORT_FIGURES_DIR}

print(f"\n\u2713 Pipeline completata al 100%! I grafici e i riassunti CSV ti aspettano in: {REPORT_FIGURES_DIR}")

--- Generazione Report Analitici e Grafici delle Performance ---

✓ Pipeline completata al 100%! I grafici e i riassunti CSV ti aspettano in: reports/figures
CPU times: total: 31.2 ms
Wall time: 4.87 s


2026-06-25 15:32:41,054 | INFO | generated new fontManager


In [16]:
import json
from pathlib import Path

# Percorso del file delle preferenze DPO generato dalla pipeline
file_path = "data/final/test_preferences.jsonl"

if not Path(file_path).exists():
    print(f"⚠️ Attenzione: Il file '{file_path}' non esiste ancora.")
    print("Assicurati di aver completato con successo la cella del 'build_preferences.py'.")
else:
    print("🏆 ================================================== 🏆")
    print("     LETTERATURA COMICA: TUTTE LE FRASI VINCENTI (CHOSEN)     ")
    print("🏆 ================================================== 🏆\n")
    
    conteggio = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            # Carica la riga come dizionario JSON
            data = json.loads(line)
            
            # Estrazione dei metadati chiave
            record_id = data.get("id", "N/A")
            frase_vincente = data.get("chosen", "N/A")
            modello_autore = data.get("source_models", {}).get("chosen", "Sconosciuto")
            modello_giudice = data.get("judge", "Sconosciuto")
            
            # Stampa formattata e pulita nella console del notebook
            print(f"🆔 [{record_id}]")
            print(f"✍️ Autore: {modello_autore.upper()}  |  ⚖️ Arbitro del Match: {modello_giudice.upper()}")
            print(f"🎤 Battuta: \"{frase_vincente}\"")
            print("-" * 70)
            
            conteggio += 1
            
    print(f"\n✓ Visualizzazione completata. Stampate {conteggio} preferenze reali.")

🏆 ================================================== 🏆
     LETTERATURA COMICA: TUTTE LE FRASI VINCENTI (CHOSEN)     
🏆 ================================================== 🏆

🆔 [en_0001]
✍️ Autore: LLAMA  |  ⚖️ Arbitro del Match: MISTRAL
🎤 Battuta: "Ryanair's just trying to keep Spain from getting too comfortable – it's all about preserving the country's national pastime: anxiety travel."
----------------------------------------------------------------------
🆔 [en_0002]
✍️ Autore: LLAMA  |  ⚖️ Arbitro del Match: MISTRAL
🎤 Battuta: "It's finally back where it belongs – in the Nazis' art collection, now running a very successful Airbnb."
----------------------------------------------------------------------
🆔 [en_0003]
✍️ Autore: QWEN  |  ⚖️ Arbitro del Match: MISTRAL
🎤 Battuta: "So, they finally realized childcare needs reform... because kids still cry."
----------------------------------------------------------------------
🆔 [en_0004]
✍️ Autore: QWEN  |  ⚖️ Arbitro del Match: MISTRAL
🎤 

In [ ]:
import json
from pathlib import Path

file_judgments = "data/judged/rag_test_judgments.jsonl"

if not Path(file_judgments).exists():
    print(f"⚠️ File '{file_judgments}' non trovato!")
else:
    best_jokes = []

    with open(file_judgments, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            record_id = data.get("id")
            judge = data.get("judge")
            scores = data.get("scores", {})
            
            # Estraiamo i voti dell'umorismo per il Modello A e il Modello B
            humor_scores = scores.get("humor", {})
            score_a = int(humor_scores.get("A", 0))
            score_b = int(humor_scores.get("B", 0))
            
            # Salviamo la battuta A se ha un punteggio alto (>= 4)
            best_jokes.append({
                "id": record_id,
                "joke": data.get("joke_a"),
                "author": data.get("metadata", {}).get("blind_labels", {}).get("A", "N/A"),
                "humor_rating": score_a,
                "judge": judge
            })
            
            # Salviamo la battuta B se ha un punteggio alto (>= 4)
            best_jokes.append({
                "id": record_id,
                "joke": data.get("joke_b"),
                "author": data.get("metadata", {}).get("blind_labels", {}).get("B", "N/A"),
                "humor_rating": score_b,
                "judge": judge
            })

    # Ordiniamo tutte le battute del dataset dal voto più alto al più basso
    best_jokes = sorted(best_jokes, key=lambda x: x["humor_rating"], reverse=True)

    print("🏆 ======================================================= 🏆")
    print("     THE LAUGH FACTORY: LE BATTUTE PIÙ DIVERTENTI DEL TORNEO    ")
    print("🏆 ======================================================= 🏆\n")

    # Stampiamo le prime 5 battute sul podio (o quelle con voto eccellente)
    for i, item in enumerate(best_jokes[:5]):
        print(f"🥇 Posizione #{i+1} | Voto Umorismo: {item['humor_rating']}/5 ⭐")
        print(f"🆔 ID Notizia: {item['id']}")
        print(f"✍️ Autore: {item['author'].upper()}  |  ⚖️ Giudicato da: {item['judge'].upper()}")
        print(f"🎤 Battuta: \"{item['joke']}\"")
        print("-" * 75)